In [ ]:
import io, zipfile, requests
from pathlib import Path

SOURCES = {
    "ipl":  "https://cricsheet.org/downloads/ipl_json.zip",
    "t20i": "https://cricsheet.org/downloads/t20s_json.zip",
}
RAW_DIR = Path("/content/data/raw")

for name, url in SOURCES.items():
    out = RAW_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {name} ...")
    r = requests.get(url, timeout=300); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        zf.extractall(out)
    print(f"  -> {len(list(out.glob('*.json')))} files")

  -> 1243 files
  -> 5524 files


In [ ]:
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd

DEATH_START = 15
MIN_PRIOR_BALLS = 30

dated = []
for src in Path("/content/data/raw").iterdir():
    for p in src.glob("*.json"):
        try:
            with open(p, encoding="utf-8") as f:
                m = json.load(f)
            if m["info"].get("gender") != "male":
                continue
            dated.append((m["info"]["dates"][0], p.stem, m))
        except Exception:
            pass
dated.sort(key=lambda x: x[0])
print(f"{len(dated)} matches, {dated[0][0]} -> {dated[-1][0]}")

bat = defaultdict(lambda: {"balls":0,"runs":0,"dots":0,"boundaries":0})
bowl = defaultdict(lambda: {"balls":0,"runs":0,"dots":0,"wickets":0})
rows = []

for date, match_id, m in dated:
    for inn_idx, innings in enumerate(m.get("innings", [])):
        target = innings.get("target", {}).get("runs")
        runs_inn, balls_inn = 0, 0                       # <-- NEW: innings score state
        for over in innings.get("overs", []):
            dels = over["deliveries"]
            if not dels:
                continue
            if over["over"] >= DEATH_START:
                striker, bowler = dels[0]["batter"], dels[0]["bowler"]
                bs, ws = bat[striker], bowl[bowler]
                if bs["balls"] >= MIN_PRIOR_BALLS and ws["balls"] >= MIN_PRIOR_BALLS:
                    striker_runs = sum(d["runs"]["batter"] for d in dels if d["batter"] == striker)
                    over_runs = sum(d["runs"]["total"] for d in dels)
                    rows.append({
                        "date": date, "match_id": match_id,
                        "over": over["over"] + 1, "innings": inn_idx + 1,
                        "batter": striker, "bowler": bowler,
                        "bat_prior_balls": bs["balls"],
                        "bat_death_sr": round(100*bs["runs"]/bs["balls"], 2),
                        "bat_boundary_pct": round(100*bs["boundaries"]/bs["balls"], 2),
                        "bat_dot_pct": round(100*bs["dots"]/bs["balls"], 2),
                        "bowl_prior_balls": ws["balls"],
                        "bowl_death_econ": round(6*ws["runs"]/ws["balls"], 2),
                        "bowl_dot_pct": round(100*ws["dots"]/ws["balls"], 2),
                        "bowl_wkt_per_over": round(6*ws["wickets"]/ws["balls"], 3),
                        "is_chase": 1 if (inn_idx == 1 and target) else 0,
                        "runs_needed": (target - runs_inn) if (inn_idx == 1 and target) else 0,   # NEW
                        "balls_remaining": 120 - balls_inn,                                        # NEW
                        "batter_12plus": 1 if striker_runs >= 12 else 0,
                        "bowler_max8": 1 if over_runs <= 8 else 0,
                    })
            for d in dels:
                ex = d.get("extras", {})
                legal = not ("wides" in ex or "noballs" in ex)
                rb = d["runs"]["batter"]
                b = bat[d["batter"]]
                if "wides" not in ex:
                    b["balls"] += 1; b["runs"] += rb
                    if rb == 0: b["dots"] += 1
                    if rb in (4, 6): b["boundaries"] += 1
                w = bowl[d["bowler"]]
                if legal:
                    w["balls"] += 1
                    if d["runs"]["total"] == 0: w["dots"] += 1
                w["runs"] += d["runs"]["total"] - ex.get("byes", 0) - ex.get("legbyes", 0)
                for wk in d.get("wickets", []):
                    if wk.get("kind") not in ("run out", "retired hurt", "obstructing the field"):
                        w["wickets"] += 1
                runs_inn += d["runs"]["total"]           # <-- NEW
                if legal: balls_inn += 1                 # <-- NEW

df = pd.DataFrame(rows)
df.to_csv("/content/death_overs.csv", index=False)
print(f"{len(df):,} death overs | 12+ rate {df.batter_12plus.mean():.1%} | <=8 rate {df.bowler_max8.mean():.1%}")

4687 matches, 2005-02-17 -> 2026-07-02
30,008 death overs | 12+ rate 9.6% | <=8 rate 50.2%


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
import joblib

df = pd.read_csv("/content/death_overs.csv", parse_dates=["date"])

FEATURES = ["over", "innings", "is_chase", "runs_needed", "balls_remaining",
            "bat_prior_balls", "bat_death_sr", "bat_boundary_pct", "bat_dot_pct",
            "bowl_prior_balls", "bowl_death_econ", "bowl_dot_pct", "bowl_wkt_per_over"]

cutoff = df["date"].max() - pd.DateOffset(years=2)
train, test = df[df.date < cutoff], df[df.date >= cutoff]

models = {}
for target in ["batter_12plus", "bowler_max8"]:
    m = HistGradientBoostingClassifier(max_depth=5, learning_rate=0.07,
                                       max_iter=350, random_state=42)
    m.fit(train[FEATURES], train[target])
    p = m.predict_proba(test[FEATURES])[:, 1]
    models[target] = m
    print(f"{target:15s} AUC={roc_auc_score(test[target], p):.4f}  "
          f"Brier={brier_score_loss(test[target], p):.4f}")

batter_12plus   AUC=0.6554  Brier=0.0774
bowler_max8     AUC=0.6731  Brier=0.2251


In [ ]:
bat_cols = ["bat_prior_balls","bat_death_sr","bat_boundary_pct","bat_dot_pct"]
bowl_cols = ["bowl_prior_balls","bowl_death_econ","bowl_dot_pct","bowl_wkt_per_over"]

bat_stats = (df.sort_values("date").groupby("batter").tail(1)
               .set_index("batter")[bat_cols])
bowl_stats = (df.sort_values("date").groupby("bowler").tail(1)
                .set_index("bowler")[bowl_cols])

joblib.dump({"models": models, "features": FEATURES,
             "bat_stats": bat_stats, "bowl_stats": bowl_stats},
            "/content/matchup_app_bundle.joblib")
print(f"Saved bundle: {len(bat_stats)} batters, {len(bowl_stats)} bowlers")

Saved bundle: 2251 batters, 2196 bowlers


In [ ]:
recent = df[df.date >= df.date.max() - pd.DateOffset(years=3)]

print("Hardest bowlers to score off at the death (min 40 death overs):")
g = recent.groupby("bowler").agg(overs=("batter_12plus","size"),
                                 conceded_12plus=("batter_12plus","mean"))
print(g[g.overs >= 40].sort_values("conceded_12plus").head(10).round(3))

print("\nMost explosive death batters (min 40 death overs):")
g = recent.groupby("batter").agg(overs=("batter_12plus","size"),
                                 hit_12plus=("batter_12plus","mean"))
print(g[g.overs >= 40].sort_values("hit_12plus", ascending=False).head(10).round(3))

Hardest bowlers to score off at the death (min 40 death overs):
                   overs  conceded_12plus
bowler                                   
JJ Bumrah             84            0.012
Rizwan Butt           76            0.013
IO Okpe               56            0.036
Ehsan Khan            51            0.039
Ali Dawood            77            0.052
Mustafizur Rahman     71            0.056
R Ngarava             50            0.060
Virandeep Singh       50            0.060
Junaid Siddique       61            0.066
PJ Cummins            45            0.067

Most explosive death batters (min 40 death overs):
                overs  hit_12plus
batter                           
Shashank Singh     42       0.262
TH David           73       0.247
DS Airee           46       0.196
T Stubbs           67       0.194
DA Miller          57       0.193
R Powell           61       0.180
RK Singh           53       0.170
MP Stoinis         41       0.146
H Klaasen          55       0.145
Tilak 